# FD001 Replication: CNN-LSTM va BiLSTM-Attn

Notebook nay chay replication FD001, luu artifact sau moi lan train, va tao bang so sanh voi paper.

Output chinh:
- `results/metrics.csv`
- `results/comparison_fd001.csv`
- `results/runs/<run_id>/metrics.json`
- `results/runs/<run_id>/predictions.csv`
- `results/runs/<run_id>/training_history.csv`
- `results/runs/<run_id>/checkpoint.pt` cho PyTorch models

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

PYTHON = PROJECT_ROOT / '.venv-codex' / 'Scripts' / 'python.exe'
if not PYTHON.exists():
    PYTHON = Path(sys.executable)

RESULTS_DIR = PROJECT_ROOT / 'results'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)
print('Python:', PYTHON)

In [ ]:
from rul_replication.data import load_cmapss

data = load_cmapss('FD001', window_size=30, clip_rul=125)
print('train_windows:', data.train_windows.shape)
print('train_labels:', data.train_labels.shape)
print('test_windows:', data.test_windows.shape)
print('test_labels:', data.test_labels.shape)
print('features:', data.feature_names)

In [ ]:
def run_experiment(model, epochs=80, batch_size=128, run_name=None):
    cmd = [
        str(PYTHON), str(PROJECT_ROOT / 'run_experiment.py'),
        '--fd', 'FD001',
        '--model', model,
        '--epochs', str(epochs),
        '--batch-size', str(batch_size),
        '--patience', '10',
    ]
    if run_name:
        cmd += ['--run-name', run_name]
    print('Running:', ' '.join(cmd))
    completed = subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
    return completed.returncode

## Train CNN-LSTM

FD001 CNN-LSTM tren CPU mat khoang vai phut. Ket qua duoc luu vao `results/runs/FD001_cnn_lstm_final/`.

In [ ]:
run_experiment('cnn_lstm', epochs=50, batch_size=128, run_name='FD001_cnn_lstm_final')

## Train BiLSTM-Attn

BiLSTM-Attn nang hon. Tren CPU co the mat 25-35 phut cho 80 epoch. Ket qua duoc luu vao `results/runs/FD001_bilstm_attn_final/`.

In [ ]:
run_experiment('bilstm_attn', epochs=80, batch_size=256, run_name='FD001_bilstm_attn_final')

## So sanh voi paper

In [ ]:
metrics = pd.read_csv(RESULTS_DIR / 'metrics.csv')
wanted = metrics[
    ((metrics['model'] == 'cnn_lstm') & (metrics['epochs'] == 50)) |
    ((metrics['model'] == 'bilstm_attn') & (metrics['epochs'] == 80)) |
    (metrics['model'] == 'sgd')
].copy()
wanted = wanted.drop_duplicates(subset=['fd', 'model', 'epochs', 'batch_size', 'seed'], keep='last')
wanted['paper_model_name'] = wanted['model'].map({
    'sgd': 'SGD',
    'cnn_lstm': 'CNN-LSTM',
    'bilstm_attn': 'BiLSTM-Attn',
})
cols = [
    'fd', 'paper_model_name', 'model', 'epochs', 'batch_size', 'rmse',
    'paper_rmse', 'rmse_delta_vs_paper', 'r2', 'rrmse', 'kge', 'nasa_score', 'run_id'
]
comparison = wanted[cols].sort_values(['model', 'epochs'])
comparison.to_csv(RESULTS_DIR / 'comparison_fd001.csv', index=False)
comparison

## Xem prediction cua run moi nhat

In [ ]:
for run_name in ['FD001_cnn_lstm_final', 'FD001_bilstm_attn_final']:
    pred_path = RESULTS_DIR / 'runs' / run_name / 'predictions.csv'
    if pred_path.exists():
        pred = pd.read_csv(pred_path)
        ax = pred.plot(x='unit', y=['y_true', 'y_pred'], figsize=(10, 4), title=run_name)
        ax.set_ylabel('RUL')
        plt.show()
        display(pred.head())
    else:
        print('Missing:', pred_path)